# Section 2 · Notebook 1 — Building Tools

1. From function to tool — turning a Python function into a tool with `@tool`
2. Your first tool-enabled agent — a tip calculator
3. When one tool isn't enough — a multi-tool sales assistant
4. Building custom tools — an inventory checker

In [2]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from strands.models.bedrock import BedrockModel

MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "amazon.nova-lite-v1:0")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
print(f"Using {MODEL_ID} in {REGION}")

Using amazon.nova-lite-v1:0 in us-east-1


---
## 1. From Function to Tool

Here is an ordinary Python function. It works when *you* call it, but an agent has no way to know it exists:

In [4]:
import requests


def check_server_status_plain(server_url):
    """Check if a server is responding."""
    try:
        response = requests.get(server_url, timeout=5)
        return f"Server is up. Status code: {response.status_code}"
    except requests.exceptions.RequestException:
        return "Server is down or unreachable"


print(check_server_status_plain("https://httpbin.org/get"))

Server is up. Status code: 200


Add three things and it becomes a tool the agent can discover and call:
- the `@tool` decorator
- type hints
- a clear docstring (with an `Args:` section)

In [5]:
from strands import Agent, tool


@tool  # Decorator
def check_server_status(server_url: str) -> str:  # Type hints
    """Check if a server is responding by making an HTTP request.

    Args:
        server_url: The URL of the server to check

    Returns:
        A message indicating whether the server is up or down
    """
    try:
        response = requests.get(server_url, timeout=5)
        return f"Server is up. Status code: {response.status_code}"
    except requests.exceptions.RequestException:
        return "Server is down or unreachable"

In [5]:
agent = Agent(model=model, tools=[check_server_status])
response = agent("Is the staging server running? Check https://httpbin.org/get")

Sure! Let me check the staging server for you right now.
Tool #1: check_server_status
The staging server at **https://httpbin.org/get** is **up and running**! ✅ It responded with a **200 OK** status code, meaning it's healthy and accessible. No issues to report!

---
## 2. Your First Tool-Enabled Agent — Tip Calculator

In [6]:
@tool
def calculate_tip(bill_amount: float, tip_percentage: float, num_people: int = 1) -> dict:
    """Calculate tip and split the bill among people.

    Args:
        bill_amount: Total bill amount in dollars
        tip_percentage: Tip percentage (e.g., 15, 18, 20)
        num_people: Number of people splitting the bill (default: 1)
    """
    tip = bill_amount * (tip_percentage / 100)
    total = bill_amount + tip
    per_person = total / num_people

    return {
        "bill": bill_amount,
        "tip": round(tip, 2),
        "total": round(total, 2),
        "per_person": round(per_person, 2),
    }


agent = Agent(model=model, tools=[calculate_tip])

response = agent(
    "The bill is $85. What's a 20% tip, and how much does each person pay if we're splitting it 4 ways?"
)
print("\n\nFinal answer:\n", response.message["content"][0]["text"])

<thinking>To answer this question, I need to calculate the tip amount based on the given bill and tip percentage. Then, I need to calculate the total amount including the tip and finally divide it by the number of people to get the amount each person needs to pay.</thinking>

Tool #1: calculate_tip
<thinking>Based on the result from the tool, I have all the information needed to answer the question. The tip is $17 and each person pays $25.5 if the bill is split among 4 people.</thinking>

The 20% tip on a bill of $85 is $17. If you're splitting the bill among 4 people, each person would pay $25.5.

Final answer:
 <thinking>Based on the result from the tool, I have all the information needed to answer the question. The tip is $17 and each person pays $25.5 if the bill is split among 4 people.</thinking>

The 20% tip on a bill of $85 is $17. If you're splitting the bill among 4 people, each person would pay $25.5.


Try different phrasings — the agent understands intent, not just keywords:

In [7]:
agent("What's a 15% tip on $42?")
agent("Bill is $120, we want to tip 18%, split between 3 people")
agent("Calculate tip for $67.50 at 20%")

<thinking>To answer this question, I need to calculate the tip amount based on the given bill and tip percentage. Then, I need to divide the total amount (bill + tip) by the number of people if specified, or just provide the tip amount if not.</thinking> 
Tool #2: calculate_tip
<thinking>Based on the result from the tool, I have all the information needed to answer the question. The 15% tip on a bill of $42 is $6.3.</thinking>

The 15% tip on a bill of $42 is $6.3.<thinking>To answer this question, I need to calculate the tip amount based on the given bill and tip percentage. Then, I need to calculate the total amount including the tip and finally divide it by the number of people to get the amount each person needs to pay.</thinking> 
Tool #3: calculate_tip
<thinking>Based on the result from the tool, I have all the information needed to answer the question. The 18% tip on a bill of $120 is $21.6. If you're splitting the bill and tip among 3 people, each person would pay $47.2.</think

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '<thinking>Based on the result from the tool, I have all the information needed to answer the question. The 20% tip on a bill of $67.5 is $13.5.</thinking>\n\nThe 20% tip on a bill of $67.5 is $13.5.'}], 'metadata': {'usage': {'inputTokens': 1457, 'outputTokens': 69, 'totalTokens': 1526}, 'metrics': {'latencyMs': 627, 'timeToFirstByteMs': 502}}, 'tracking_id': '166a0f44-64d4-460b-8e99-86bad4233006'}, metrics=EventLoopMetrics(cycle_count=8, tool_metrics={'calculate_tip': ToolMetrics(tool={'toolUseId': 'tooluse_HVeQctF4LnfKcefnHo7B4e', 'name': 'calculate_tip', 'input': {'bill_amount': 67.5, 'tip_percentage': 20, 'num_people': 1}}, call_count=4, success_count=4, error_count=0, total_time=0.003703207999933511)}, cycle_durations=[1.516036033630371, 0.8897411823272705, 0.9589660167694092, 0.8932499885559082, 0.9563896656036377, 1.0606780052185059, 0.9690761566162109, 0.8491818904876709], agent_invocations=

---
## 3. When One Tool Isn't Enough — Multi-Tool Sales Assistant

Give the agent several small tools and it works out the order to call them in. One request — "pull sales data and email a summary" — becomes three tool calls.

In [8]:
@tool
def get_sales_data(quarter: str) -> dict:
    """Retrieve sales data for a specific quarter."""
    return {"revenue": 1250000, "deals": 47, "quarter": quarter}


@tool
def analyze_sales(revenue: int, deals: int, quarter: str) -> str:
    """Calculate key metrics from sales data."""
    avg_deal = revenue / deals
    return f"{quarter}: ${revenue:,} revenue, {deals} deals, ${avg_deal:,.0f} avg deal size"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email message."""
    print(f"\n[EMAIL] to={to} | subject={subject}\n{body}\n")
    return f"Email sent to {to}"


agent = Agent(model=model, tools=[get_sales_data, analyze_sales, send_email])
response = agent("Pull last quarter's sales data and email a summary to team@company.com")

<thinking>The User wants the sales data for the last quarter and to have it emailed to a specific address. To achieve this, I need to first retrieve the sales data for the last quarter and then analyze it to create a summary email. I'll start by retrieving the sales data.</thinking>

Tool #1: get_sales_data
<thinking>I have successfully retrieved the sales data for the last quarter. The revenue was 1,250,000 and there were 47 deals. Next, I need to analyze this data to create a summary email to send to the team. I will use the 'analyze_sales' tool to calculate key metrics.</thinking> 
Tool #2: analyze_sales
<thinking>I have calculated the key metrics from the sales data. The average deal size for the last quarter was $26,596. Now, I need to draft an email with this information and send it to team@company.com. I will use the 'send_email' tool for this.</thinking> 
Tool #3: send_email

[EMAIL] to=team@company.com | subject=Sales Summary for Last Quarter
Hi Team,

Here is the summary of o

---
## 4. Building Custom Tools — Inventory Checker

When no prebuilt tool fits, build your own. This one simulates querying a product database.

In [9]:
@tool
def check_inventory(product_id: str) -> str:
    """Check if a product is in stock.

    Args:
        product_id: The product ID to check (e.g., "PROD-123")
    """
    inventory = {"PROD-123": 15, "PROD-456": 0, "PROD-789": 8}
    quantity = inventory.get(product_id, 0)

    if quantity > 0:
        return f"Product {product_id} is in stock. We have {quantity} units available."
    return f"Product {product_id} is currently out of stock."


agent = Agent(model=model, tools=[check_inventory])

# All of these work — the agent understands intent, not just keywords
agent("Is PROD-123 in stock?")
agent("Do we have PROD-456 available?")
agent("Check inventory for PROD-789")
agent("Can I order PROD-123 right now?")

<thinking>The User wants to know if product PROD-123 is in stock. The only tool available is `check_inventory`, which can be used to answer this question. I need to call this tool with the product ID as an argument.</thinking>

Tool #1: check_inventory
<thinking>I have received the result from the `check_inventory` tool. The product PROD-123 is in stock, and we have 15 units available. I can now provide this information to the User.</thinking>

The product PROD-123 is in stock. We have 15 units available.<thinking>The User wants to know if product PROD-456 is in stock. I need to call the `check_inventory` tool with the product ID as an argument to get the answer.</thinking> 
Tool #2: check_inventory
<thinking>I have received the result from the `check_inventory` tool. The product PROD-456 is currently out of stock. I can now provide this information to the User.</thinking>

Unfortunately, the product PROD-456 is currently out of stock.<thinking>The User wants to know if product PROD-78

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "<thinking>The User wants to know if they can order product PROD-123 right now. However, I don't have the information to determine if an order can be placed at this moment. The `check_inventory` tool only provides information on the stock status and quantity, not on the ability to place an order. I need to inform the User that I can't provide this information based on the available tools.</thinking>\n\nI'm sorry, but I can't provide information on whether you can order product PROD-123 right now based on the available tools. The `check_inventory` tool only provides information on the stock status and quantity, not on the ability to place an order. I recommend checking with our sales or customer service team for more information on ordering the product."}], 'metadata': {'usage': {'inputTokens': 1015, 'outputTokens': 168, 'totalTokens': 1183}, 'metrics': {'latencyMs': 1132, 'timeToFirstByteMs': 509}}, 